In [21]:
from openai import OpenAI
from tqdm import tqdm
import os
import time
import pandas as pd
import numpy as np
import json
import numpy as np
from pandas import Timestamp

df = pd.read_pickle(r'./df_filtered_comments.pkl')
df = df.drop_duplicates(subset=['comment_id'])
df = df.reset_index(drop=True)
display(df.head())
print(len(df))

,video_id,comment_id,author,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,updated_at,like_count,viewer_rating,can_rate,is_reply,parent_id,channel_id,classification
0,RvDQEbrQ780,UgwrA-UehYgp7Vdey1Z4AaABAg,@yahoo17,https://yt3.ggpht.com/rzhMWwlENKgS-TyMiy_Lga78...,http://www.youtube.com/@yahoo17,UCcIFfuAzUIiOm_o4fEqPn8A,O Sr. deveria ter ido ao último Congresso de P...,2025-10-08 02:37:58+00:00,2025-10-08 02:37:58+00:00,0,none,True,False,None,UCfeIEwYYNEnEUDirv6KpoAQ,sim
1,5aEwLfS3-xQ,UgxiM_N6HXks8A41rWF4AaABAg,@cm6765,https://yt3.ggpht.com/ytc/AIdro_me9MbJebwUeuLw...,http://www.youtube.com/@cm6765,UCcrnXRWFomgu7VYBGUgBlEQ,Muito posetivo \nEssas vacinas vão ajudar a es...,2025-09-27 12:02:33+00:00,2025-09-27 12:02:33+00:00,1,none,True,False,None,UCFsS7qMsi7OLtsORePv0axg,sim
2,KhB2pY8-Jbk,UgyjbKcCM_OVq_7ZDPt4AaABAg,@albertobeto5362,https://yt3.ggpht.com/ytc/AIdro_m25_CWV9Iq-V5a...,http://www.youtube.com/@albertobeto5362,UCYSQWyLp2tFaSkzdGFJSIBw,Não é vacina mas o responsável em publicar o v...,2025-09-25 04:41:18+00:00,2025-09-25 04:41:18+00:00,0,none,True,False,None,UCXdXYG8dUmEv6jhEji_lSHg,sim
3,U2DsHbjPg4I,UgzgGHSqjTXbT5apB7B4AaABAg,@meunomeepablo412,https://yt3.ggpht.com/KQMZ5bTUIIQQgiLV-nZOodEB...,http://www.youtube.com/@meunomeepablo412,UCYtvrANMXBcJLlqzm2hgjgw,Algumas dessas tais vacinas da covide já foram...,2025-10-01 13:02:25+00:00,2025-10-01 13:03:00+00:00,0,none,True,False,None,UCtPLzMdFXwcQ6VE3ZBk2DUw,sim
4,U2DsHbjPg4I,Ugx7pV7xUHxCvdYZAKp4AaABAg,@Luxzy_444,https://yt3.ggpht.com/-PrmbdbI-YJYMIBstlvXkdxn...,http://www.youtube.com/@Luxzy_444,UCKVrXqrpzgiZuk_SRZgrUXw,"Galera usando esteroides, energético, cigarro ...",2025-09-30 16:54:20+00:00,2025-09-30 16:54:20+00:00,0,none,True,False,None,UCtPLzMdFXwcQ6VE3ZBk2DUw,sim


48209


In [22]:

# função utilitária para tornar valores JSON-serializáveis
def make_serializable(val):
    # pandas Timestamp
    if isinstance(val, Timestamp):
        return val.isoformat()
    # numpy integer/float/bool
    if isinstance(val, (np.integer,)):
        return int(val)
    if isinstance(val, (np.floating,)):
        return float(val)
    if isinstance(val, (np.bool_,)):
        return bool(val)
    # pandas / numpy NA
    try:
        # pd.isna handles many NA types
        import pandas as _pd
        if _pd.isna(val):
            return None
    except Exception:
        pass
    # estruturas conversíveis (list, dict, str, int, float, bool) passam direto
    return val

In [24]:
OUTPUT_JSON = 'stance_detection_results.jsonl'
OUTPUT_LOG = 'stance_detection_results.txt'
MODEL = 'gpt-4o-mini'

PROMPT_BASE = """
Você irá analisar comentários do YouTube relacionados a vacinas. Sua tarefa é classificar o posicionamento/sentimento do autor do comentário em relação à vacinação.

Rótulos possíveis:
- Positivo: o comentário expressa apoio, confiança, incentivo ou defesa da vacinação.
- Negativo: o comentário demonstra rejeição, desconfiança, crítica ou oposição à vacinação.
- Neutro: o comentário é ambíguo, irrelevante para opinião sobre vacinas, ou não deixa claro o posicionamento do autor.

Instruções:
Leia o comentário fornecido. Retorne a classificação com base no conteúdo explícito ou claramente implícito sobre vacinação.
Formato de saída:
<Positivo | Negativo | Neutro>

Comentário:
"""
#print(PROMPT_BASE)


done_ids = set()
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                done_ids.add(record['comment_id'])
            except Exception:
                pass

print(len(done_ids))
client = OpenAI()
for i in tqdm(range(len(df))):
    row = df.iloc[i]
    if row['comment_id'] in done_ids:
        continue
    
    #print(row['comment_id'])

    comment = f'<{row["comment"]}>'
    full_prompt = PROMPT_BASE + comment
    #print(full_prompt)
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            service_tier="priority",
            messages=[{'role' : 'user', 'content' : full_prompt}],
            temperature=0
        )
        model_response = response.choices[0].message.content.strip().lower()

        with open(OUTPUT_LOG, 'a', encoding='utf-8') as f_log:
            f_log.write(f'Idx: {i} -- {row["comment_id"]}')
            f_log.write(f'{full_prompt}\n')
            f_log.write(f'{model_response}\n')
            f_log.write(f'---------------\n')
        reply = ''
        for line in model_response.split('\n'):
            line = line.strip().lower()
            if 'positivo' in line:
                reply = 'positivo'
            elif 'negativo' in line:
                reply = 'negativo'
            elif 'neutro' in line:
                reply = 'neutro'
        row_dict = row.to_dict()
        serializable_row = {k:make_serializable(v) for k,v in row_dict.items()}
        res = {
            **serializable_row,
            'stance_detection' : reply
        }

        with open(OUTPUT_JSON, 'a', encoding='utf-8') as f_json:
            json.dump(res, f_json, ensure_ascii=False)
            f_json.write('\n')
        
        
    
    except Exception as e:
        print(f'Erro {e}')
    
    

48209


100%|█████████████████████████████████████████████████████████████████████████| 48209/48209 [00:03<00:00, 12407.38it/s]


In [12]:
len(done_ids)

48209